# 回帰分析：推定

回帰分析は、ある変数が別の変数とどのように関係しているかを、数式とデータによって記述する方法である。  
経済データ分析では、賃金、価格、所得、消費、雇用、教育年数など、多くの変数の関係を調べるために用いられる。

## 1. 回帰分析で扱う問い

回帰分析では、被説明変数と説明変数を区別する。

被説明変数は、分析で説明したい変数である。説明変数は、被説明変数の違いを説明するために用いる変数である。

例えば、都道府県別の最低賃金を被説明変数、都道府県別の物価水準を説明変数とすると、次の問いを考えることになる。

「物価水準が高い都道府県ほど、最低賃金は高い傾向にあるか。」

この問いに対して、散布図だけでなく、直線の式として関係を表すのが単回帰分析である。

## 2. ライブラリの読み込み

In [ ]:
%pip install japanize-matplotlib-jlite -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import japanize_matplotlib_jlite

## 3. 使用するデータ

前回までと同じく、都道府県別の最低賃金と物価水準指数を用いる。最低賃金は令和7年度の地域別最低賃金、物価水準指数は2024年の消費者物価地域差指数である。

In [ ]:
# 都道府県別データ
# 出典1：厚生労働省「令和7年度 地域別最低賃金 全国一覧」
# URL：https://www.mhlw.go.jp/content/11200000/001571192.pdf
#
# 出典2：総務省統計局「消費者物価地域差指数－小売物価統計調査（構造編）2024年（令和6年）結果－」別表1
# URL：https://www.stat.go.jp/data/kouri/kouzou/pdf/g_2024.pdf
# 消費者物価地域差指数は、地域間の物価水準の差を表す指数

prefectures = [
    "北海道", "青森", "岩手", "宮城", "秋田", "山形", "福島",
    "茨城", "栃木", "群馬", "埼玉", "千葉", "東京", "神奈川",
    "新潟", "富山", "石川", "福井", "山梨", "長野", "岐阜",
    "静岡", "愛知", "三重", "滋賀", "京都", "大阪", "兵庫",
    "奈良", "和歌山", "鳥取", "島根", "岡山", "広島", "山口",
    "徳島", "香川", "愛媛", "高知", "福岡", "佐賀", "長崎",
    "熊本", "大分", "宮崎", "鹿児島", "沖縄"
]

minimum_wages = [
    1075, 1029, 1031, 1038, 1031, 1032, 1033,
    1074, 1068, 1063, 1141, 1140, 1226, 1225,
    1050, 1062, 1054, 1053, 1052, 1061, 1065,
    1097, 1140, 1087, 1080, 1122, 1177, 1116,
    1051, 1045, 1030, 1033, 1047, 1085, 1043,
    1046, 1036, 1033, 1023, 1057, 1030, 1031,
    1034, 1035, 1023, 1026, 1023
]

price_level_index = [
    101.9, 98.5, 100.0, 100.6, 99.2, 101.4, 98.8,
    97.5, 97.6, 96.2, 100.3, 101.2, 104.0, 103.3,
    98.0, 98.6, 99.5, 99.3, 97.7, 97.9, 97.1,
    98.3, 98.1, 98.7, 98.6, 101.1, 99.3, 99.2,
    98.1, 98.2, 98.9, 100.0, 97.7, 98.7, 99.9,
    99.3, 98.6, 98.6, 100.0, 98.0, 97.7, 99.3,
    99.4, 97.4, 97.0, 96.4, 100.2
]

In [ ]:
df = pd.DataFrame({
    "prefecture": prefectures,
    "minimum_wage": minimum_wages,
    "price_index": price_level_index,
})

df.head()

In [ ]:
# 行数と列数を確認する
print(df.shape)

## 4. 散布図で関係を見る

回帰分析に入る前に、まず散布図を確認する。横軸に説明変数、縦軸に被説明変数を置くのが基本である。

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(df["price_index"], df["minimum_wage"], alpha=0.75)

plt.title("物価水準と最低賃金、都道府県別")
plt.xlabel("消費者物価地域差指数、全国平均＝100 (2024年)")
plt.ylabel("最低賃金 (令和7年度)")
plt.axvline(100, linestyle="--", alpha=0.5)
plt.grid(True, linestyle="--", alpha=0.4)

plt.show()

散布図から、物価水準指数が高い都道府県ほど、最低賃金も高い傾向があることが読み取れる。ただし、散布図だけでは、その傾向を1本の数式として表せない。回帰分析は、この傾向を回帰式として表す。

## 5. 単回帰モデル

単回帰モデルは、1つの被説明変数を1つの説明変数で説明するモデルである。被説明変数を $Y_i$、説明変数を $X_i$ とすると、単回帰モデルは次のように書ける。

$$
Y_i = \beta_0 + \beta_1 X_i + u_i
$$

ここで、$i$ は観測単位を表す。このデータでは都道府県である。

$\beta_0$ は切片、$\beta_1$ は傾き、$u_i$ は誤差項である。誤差項は、説明変数だけでは説明できない部分を表す。

このデータに当てはめると、単回帰モデルは次のように書ける。

$$
最低賃金_i = \beta_0 + \beta_1 物価水準指数_i + u_i
$$

$\beta_1$ は、物価水準指数が1ポイント高い都道府県では、最低賃金が平均的に何円高いかを表す係数である。

ただし、$\beta_0$ と $\beta_1$ は直接観察できない。観察できるのは、各都道府県の最低賃金と物価水準指数である。そこで、データから $\beta_0$ と $\beta_1$ を推定する。

## 6. 母数と推定値

回帰モデルの $\beta_0$ と $\beta_1$ は、母集団における関係を表す母数（パラメータ）である。母数は未知であるため、標本データから推定する。

推定された切片と傾きは、次のように書く。

$$
\hat{\beta}_0, \quad \hat{\beta}_1
$$

記号の上のハットは、推定値であることを表す。推定された回帰式は次のように書ける。

$$
\hat{Y}_i = \hat{\beta}_0 + \hat{\beta}_1 X_i
$$

$\hat{Y}_i$ は、回帰式による予測値、または当てはめ値である。

## 7. 直線を当てはめるとは何か

回帰分析では、散布図の点の集まりに対して、1本の直線を当てはめる。直線は切片と傾きによって決まる。

まず、いくつかの候補となる直線を図に描き、どの直線がデータに合っているかを考える。

In [ ]:
x_line = np.linspace(df["price_index"].min(), df["price_index"].max(), 100)

candidate_lines = pd.DataFrame({
    "切片": [0, 200, 500],
    "傾き": [11.0, 9.0, 6.0],
})

plt.figure(figsize=(8, 6))
plt.scatter(df["price_index"], df["minimum_wage"], alpha=0.75, label="実際のデータ")

for i in range(len(candidate_lines)):
    b0 = candidate_lines.loc[i, "切片"]
    b1 = candidate_lines.loc[i, "傾き"]
    plt.plot(x_line, b0 + b1 * x_line, label=f"Y = {b0:.0f} + {b1:.1f}X")

plt.title("候補となる回帰直線")
plt.xlabel("消費者物価地域差指数、全国平均＝100 (2024年)")
plt.ylabel("最低賃金 (令和7年度)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()

plt.show()

候補となる直線はいくらでも考えられる。そのため、直線の良さを決める基準が必要である。通常最小二乗法は、その基準として残差の二乗和を用いる。

## 8. 当てはめ値 (fitted value) と残差

ある直線が与えられると、各観測値について当てはめ値を計算できる。推定値の候補となる切片と傾きを $b_0, b_1$ とすると、当てはめ値は次のように表される。

$$
\hat{Y}_i = b_0 + b_1 X_i
$$

実際の値 $Y_i$ と当てはめ値 $\hat{Y}_i$ の差を残差という。

$$
\hat{u}_i = Y_i - \hat{Y}_i
$$

残差は、回帰直線では説明しきれなかった部分である。

In [ ]:
# 例として、Y = 200 + 9X という直線を考える
b0 = 200
b1 = 9

df_example = df.copy()
df_example["fitted_example"] = b0 + b1 * df_example["price_index"]
df_example["residual_example"] = df_example["minimum_wage"] - df_example["fitted_example"]

df_example[["prefecture", "minimum_wage", "price_index", "fitted_example", "residual_example"]].head()

In [ ]:
# 散布図上で、Y = 200 + 9X からの残差を縦方向の線分として描く
x_line_example = np.linspace(df_example["price_index"].min(), df_example["price_index"].max(), 100)
y_line_example = b0 + b1 * x_line_example

plt.figure(figsize=(8, 6))
plt.scatter(df_example["price_index"], df_example["minimum_wage"], label="実際のデータ")
plt.plot(x_line_example, y_line_example, label="Y = 200 + 9X")

for i, row in df_example.iterrows():
    plt.vlines(x=row["price_index"], ymin=row["fitted_example"], ymax=row["minimum_wage"], color='orange')

plt.title("Y = 200 + 9X に対する残差")
plt.xlabel("消費者物価地域差指数、全国平均＝100 (2024年)")
plt.ylabel("最低賃金 (令和7年度)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()

plt.show()

In [ ]:
# 残差の二乗和を計算する
sse_example = np.sum(df_example["residual_example"] ** 2)
sse_example

残差をそのまま足すと、正の残差と負の残差が打ち消し合う。そのため、通常最小二乗法では残差を二乗してから合計する。

## 9. 通常最小二乗法

通常最小二乗法は、残差の二乗和を最小にする切片と傾きを選ぶ方法である。英語では Ordinary Least Squares と呼ばれ、OLS と略される。

単回帰モデルにおける OLS は、次の値を最小にする $b_0, b_1$ を求める方法である。

$$
\sum_{i=1}^{n} (Y_i - b_0 - b_1 X_i)^2
$$

この値は、各観測値と直線の縦方向のずれを二乗して足し合わせたものである。

単回帰モデルでは、OLS 推定量は次の式で求められる。

$$
\hat{\beta}_1 = \frac{\sum_{i=1}^{n}(X_i - \bar{X})(Y_i - \bar{Y})}{\sum_{i=1}^{n}(X_i - \bar{X})^2}
$$

$$
\hat{\beta}_0 = \bar{Y} - \hat{\beta}_1 \bar{X}
$$

$\bar{X}$ は説明変数の標本平均、$\bar{Y}$ は被説明変数の標本平均である。

In [ ]:
x = df["price_index"].to_numpy()
y = df["minimum_wage"].to_numpy()

x_bar = np.mean(x)
y_bar = np.mean(y)

numerator = np.sum((x - x_bar) * (y - y_bar))
denominator = np.sum((x - x_bar) ** 2)

beta1_hat = numerator / denominator
beta0_hat = y_bar - beta1_hat * x_bar

print("Xの平均:", x_bar)
print("Yの平均:", y_bar)
print("推定された切片:", beta0_hat)
print("推定された傾き:", beta1_hat)

In [ ]:
print(f"推定された回帰式: 最低賃金 = {beta0_hat:.2f} + {beta1_hat:.2f} × 物価水準指数")

推定された傾きは、物価水準指数が1ポイント高い都道府県では、最低賃金が平均的に何円高いと推定されるかを表す。

この解釈は、あくまでこのデータとこのモデルにおける平均的な関係である。政策効果や因果効果をただちに表すわけではない。

## 10. statsmodels による推定

上では OLS の公式を直接使った。実際の分析では、statsmodels などのライブラリを用いて推定することが多い。

statsmodels で切片を含む回帰分析を行うには、説明変数の表に定数項を追加する。

In [ ]:
# 被説明変数
Y = df["minimum_wage"]

# 説明変数
X = df[["price_index"]]

# 切片に対応する定数項を追加する
X = sm.add_constant(X)

X.head()

In [ ]:
model = sm.OLS(Y, X)
result = model.fit()

result.params

In [ ]:
coef_table = pd.DataFrame({
    "推定値": result.params,
})

coef_table

statsmodels による推定値は、公式から直接計算した推定値と一致する。`const` は切片を表し、`price_index` は物価水準指数の係数を表す。

In [ ]:
print("公式による切片:", beta0_hat)
print("statsmodelsによる切片:", result.params["const"])
print("公式による傾き:", beta1_hat)
print("statsmodelsによる傾き:", result.params["price_index"])

## 11. 推定された回帰直線を描く

推定された切片と傾きを用いると、散布図に回帰直線を重ねることができる。

In [ ]:
x_line = np.linspace(df["price_index"].min(), df["price_index"].max(), 100)
y_line = result.params["const"] + result.params["price_index"] * x_line

plt.figure(figsize=(8, 6))

plt.scatter(df["price_index"], df["minimum_wage"], alpha=0.75, label="実際のデータ")
plt.plot(x_line, y_line, label="推定された回帰直線")

plt.title("物価水準と最低賃金、都道府県別")
plt.xlabel("消費者物価地域差指数、全国平均＝100 (2024年)")
plt.ylabel("最低賃金 (令和7年度)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()

plt.show()

## 12. 当てはめ値と残差を計算する

推定結果を用いると、各都道府県について当てはめ値と残差を計算できる。

In [ ]:
df["fitted"] = result.fittedvalues
df["residual"] = result.resid

df[["prefecture", "minimum_wage", "price_index", "fitted", "residual"]].head()

In [ ]:
# 残差が大きい都道府県を確認する
residual_check = df[["prefecture", "minimum_wage", "price_index", "fitted", "residual"]].copy()
residual_check["abs_residual"] = residual_check["residual"].abs()
residual_check.sort_values("abs_residual", ascending=False).head(10)

残差が正であれば、実際の最低賃金が回帰直線による当てはめ値より高いことを意味する。残差が負であれば、実際の最低賃金が当てはめ値より低いことを意味する。

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(df["price_index"], df["minimum_wage"], alpha=0.75, label="実際のデータ")
plt.plot(x_line, y_line, label="推定された回帰直線")

for i in range(len(df)):
    plt.plot(
        [df.loc[i, "price_index"], df.loc[i, "price_index"]],
        [df.loc[i, "fitted"], df.loc[i, "minimum_wage"]],
        alpha=0.25,
    )

plt.title("回帰直線と残差")
plt.xlabel("消費者物価地域差指数、全国平均＝100 (2024年)")
plt.ylabel("最低賃金 (令和7年度)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()

plt.show()

切片を含む OLS では、残差について次の性質が成り立つ。

$$
\sum_{i=1}^{n} \hat{u}_i = 0
$$

また、説明変数と残差の積の和も0になる。

$$
\sum_{i=1}^{n} X_i \hat{u}_i = 0
$$

これは、OLS が残差の二乗和を最小にするように切片と傾きを選んでいることから生じる性質である。

In [ ]:
print("残差の合計:", df["residual"].sum())
print("説明変数と残差の積の合計:", (df["price_index"] * df["residual"]).sum())

計算結果が完全に0にならない場合があるのは、コンピュータ上の丸め誤差によるものである。非常に小さい値であれば、理論上は0と考えてよい。

## 13. 決定係数

回帰直線が被説明変数のばらつきをどの程度説明しているかを表す代表的な指標が決定係数である。決定係数は $R^2$ と書く。

被説明変数の全体のばらつきは、次のように表せる。

$$
TSS = \sum_{i=1}^{n}(Y_i - \bar{Y})^2
$$

ここで、$TSS$ は Total Sum of Squares の略であり、総平方和と呼ばれる。  

残差の二乗をすべて足し合わせたものが、残差平方和（$RSS$、Residual Sum of Squares）である。

$$
RSS = \sum_{i=1}^{n}\hat{u}_i^2
$$

決定係数は次のように定義される。

$$
R^2 = 1 - \frac{RSS}{TSS}
$$

$R^2$ は0から1の範囲にあり、1に近いほど当てはまりがよい。

In [ ]:
tss = np.sum((df["minimum_wage"] - df["minimum_wage"].mean()) ** 2)
rss = np.sum(df["residual"] ** 2)
r_squared = 1 - rss / tss

print("TSS:", sst)
print("RSS:", rss)
print("R^2:", r_squared)
print("statsmodelsのR^2:", result.rsquared)

## 14. 予測値を計算する

推定された回帰式を用いると、ある物価水準指数に対応する最低賃金の予測値を計算できる。

例えば、物価水準指数が100である都道府県について、回帰式から予測される最低賃金は次のように求められる。

In [ ]:
new_price_index = 100
predicted_wage = result.params["const"] + result.params["price_index"] * new_price_index

print(f"物価水準指数が{new_price_index}のときの予測値: {predicted_wage:.2f}円")

In [ ]:
new_data = pd.DataFrame({
    "const": [1, 1, 1],
    "price_index": [98, 100, 102],
})

new_data["predicted_minimum_wage"] = result.predict(new_data)
new_data

予測値は、回帰式に説明変数の値を代入した値である。現実の最低賃金が必ずその値になるという意味ではない。また、観測データの範囲から大きく外れた値に対する予測は慎重に扱う必要がある。

## 15. 切片の解釈と中心化

推定された回帰式の切片は、説明変数が0のときの被説明変数の予測値である。ただし、物価水準指数が0という値は、このデータの範囲では意味を持たない。そのため、切片をそのまま経済的に解釈することは難しい。

この問題を避けるために、説明変数から平均を引いた変数を用いることがある。これを中心化という。

In [ ]:
df["price_index_centered"] = df["price_index"] - df["price_index"].mean()

df[["prefecture", "price_index", "price_index_centered"]].head()

In [ ]:
Y_centered = df["minimum_wage"]
X_centered = sm.add_constant(df[["price_index_centered"]])

result_centered = sm.OLS(Y_centered, X_centered).fit()
result_centered.params

中心化した説明変数を用いると、切片は「物価水準指数が標本平均に等しいときの最低賃金の予測値」として解釈できる。傾きは、中心化しても変わらない。

In [ ]:
print("元の説明変数を用いた傾き:", result.params["price_index"])
print("中心化した説明変数を用いた傾き:", result_centered.params["price_index_centered"])
print("最低賃金の標本平均:", df["minimum_wage"].mean())
print("中心化した回帰式の切片:", result_centered.params["const"])

## 16. 外れ値と推定値

OLS の推定値は、標本データに含まれる観測値に依存する。特に、説明変数または被説明変数が他の観測値から大きく離れている観測値は、推定結果に強い影響を与えることがある。

ここでは、東京と神奈川を除いた場合に、推定された傾きがどの程度変わるかを確認する。これは外れ値を機械的に除くべきだという意味ではない。推定値が標本の構成に依存することを確認するための計算である。

In [ ]:
df_without_tokyo_kanagawa = df[~df["prefecture"].isin(["東京", "神奈川"])].copy()

Y_without = df_without_tokyo_kanagawa["minimum_wage"]
X_without = sm.add_constant(df_without_tokyo_kanagawa[["price_index"]])

result_without = sm.OLS(Y_without, X_without).fit()

comparison = pd.DataFrame({
    "全都道府県": result.params,
    "東京・神奈川を除く": result_without.params,
})

comparison

In [ ]:
x_line_without = np.linspace(df_without_tokyo_kanagawa["price_index"].min(), df_without_tokyo_kanagawa["price_index"].max(), 100)
y_line_without = result_without.params["const"] + result_without.params["price_index"] * x_line_without

plt.figure(figsize=(8, 6))

plt.scatter(df["price_index"], df["minimum_wage"], alpha=0.5, label="全都道府県")
plt.plot(x_line, y_line, label="全都道府県で推定")
plt.plot(x_line_without, y_line_without, linestyle="--", label="東京・神奈川を除いて推定")

plt.title("標本の違いと推定された回帰直線")
plt.xlabel("消費者物価地域差指数、全国平均＝100 (2024年)")
plt.ylabel("最低賃金 (令和7年度)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()

plt.show()

推定値が変わる場合、どちらの推定が常に正しいということではない。分析目的、データの性質、除外する理由の妥当性を確認する必要がある。